---

In [1]:
# gene level

In [2]:
library(AnnotationDbi)
library(org.Hs.eg.db)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(dplyr)
library(rtracklayer)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: IRanges

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The f

## Transcript -level TSS

##### first up regulated

In [3]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/upregulated_genes_deseq.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [4]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [5]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
gene_bodies <- genes(txdb)   # direct gene coordinates, keyed by Entrez IDs
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [7]:
library(GenomicFeatures)

txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

# 1) One range per gene (Entrez IDs); ambiguous genes dropped by default
gene_bodies <- genes(txdb)                       # GRanges
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

# 2) Convert to BED6 (0-based start)
tss_df <- as.data.frame(gene_bodies)
bed_gene_df <- data.frame(
  chrom      = tss_df$seqnames,
  chromStart = tss_df$start - 1L,   # BED is 0-based, half-open
  chromEnd   = tss_df$end,
  name       = tss_df$gene_id,      # <- correct column name
  score      = 0,
  strand     = as.character(tss_df$strand),
  stringsAsFactors = FALSE
)

stopifnot(nrow(bed_gene_df) == nrow(tss_df))     # sanity check

write.table(
  bed_gene_df,
  file = "upregulated_genelevel_genebodies.bed",
  sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE
)

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [ ]:
#tss_df

In [8]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<int>,<int>,<chr>,<dbl>,<chr>
chr16,68644992,68727468,1001,0,+
chrX,101418286,101533459,100131755,0,+
chr16,31201884,31203452,100652740,0,+
chr5,138178718,138187723,10112,0,+
chr3,127480689,127537817,101927149,0,-
chr1,149923316,149927803,10262,0,-
chr22,30331987,30356919,10291,0,-
chr9,137241286,137243707,10383,0,+
chr19,41708584,41756737,1048,0,+


##### now downregulated

In [9]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/downregulated_genes_deseq.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [10]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [11]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

gene_bodies <- genes(txdb)   # direct gene coordinates, keyed by Entrez IDs
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [ ]:
#as.data.frame(gene_bodies)

In [12]:
as.data.frame(gene_bodies)

,seqnames,start,end,width,strand,gene_id
,<fct>,<int>,<int>,<int>,<fct>,<chr>
100128893,chr18,22166898,22169878,2981,-,100128893
100288152,chr5,473236,480830,7595,+,100288152
100302743,chr2,10446714,10446849,136,-,100302743
100500819,chr7,98881650,98881729,80,+,100500819
100533107,chr20,63658294,63698684,40391,+,100533107
10139,chr20,63698642,63708025,9384,-,10139
109616975,chr5,139276180,139276320,141,+,109616975
110806276,chr1,221966341,221984964,18624,+,110806276
115294,chr8,51817575,51899186,81612,-,115294


In [13]:
# gene_bodies is a GRanges from: gene_bodies <- genes(txdb)
tss_df <- as.data.frame(gene_bodies)   # cols: seqnames, start, end, strand, gene_id, ...

# Build BED6 (0-based, half-open)
bed_gene_df <- data.frame(
  chrom      = tss_df$seqnames,
  chromStart = as.integer(tss_df$start) - 1L,   # BED is 0-based
  chromEnd   = as.integer(tss_df$end),
  name       = as.character(tss_df$gene_id),    # <-- FIXED
  score      = 0,
  strand     = as.character(tss_df$strand),
  stringsAsFactors = FALSE
)

write.table(
  bed_gene_df,
  file = "downregulated_gene_level_genebodies.bed",
  sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE
)


In [14]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<int>,<int>,<chr>,<dbl>,<chr>
chr18,22166897,22169878,100128893,0,-
chr5,473235,480830,100288152,0,+
chr2,10446713,10446849,100302743,0,-
chr7,98881649,98881729,100500819,0,+
chr20,63658293,63698684,100533107,0,+
chr20,63698641,63708025,10139,0,-
chr5,139276179,139276320,109616975,0,+
chr1,221966340,221984964,110806276,0,+
chr8,51817574,51899186,115294,0,-
